# KDMAge

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.KDMAge)

class KDMAge(pyagingModel):
    """Klemera-Doubal biological age with sex-specific NHANES III parameters."""

    def __init__(self):
        super().__init__()
        for sex in ["male", "female"]:
            for name in ["q", "k", "s"]:
                self.register_buffer(f"{name}_{sex}", torch.empty(0))
            self.register_buffer(f"s_ba2_{sex}", torch.empty(0))

    def preprocess(self, x):
        """Apply BioAge's log1p transform to C-reactive protein.

        BioAge fits its ``lncrp`` biomarker against ``log1p(CRP in mg/dL)``, not
        ``ln`` as phenoage does; users supply the raw measurement so the same
        column can feed clocks that log it differently.

        Notes
        -----
        CRP is clamped to ``CRP_FLOOR_MG_DL`` first, matching ``PhenoAge``. The
        floor equals the registered lower bound, so any value it moves is already
        out of range and has already been warned about.
        """
        index = self.features.index("c_reactive_protei

In [3]:
model = pya.models.KDMAge()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "kdmage"
model.metadata["data_type"] = "clinical biomarkers"
model.metadata["species"] = "Homo sapiens"
model.metadata["year"] = 2020
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Kwon, Dayoon, and Daniel W. Belsky. \"A toolkit for quantification of biological age from blood chemistry and organ function test data: BioAge.\" GeroScience 43.6 (2021): 2795-2808."
model.metadata["doi"] = "https://doi.org/10.1007/s11357-021-00480-5"
model.metadata["research_only"] = None
model.metadata["notes"] = "Klemera-Doubal method trained sex-specifically on NHANES III (ages 30-75, non-pregnant) using the BioAge package defaults. Biomarker parameters were fit on SI-unit variants so they are natively in pyaging's unit convention."

## Download clock dependencies

The fitted parameters were extracted from the R package [dayoonkwon/BioAge](https://github.com/dayoonkwon/BioAge) and are checked in under `clocks/bioage_params/kdmage.json`. Each sex has, per biomarker, the regression of that biomarker on chronological age: `q` (intercept), `k` (slope), and `s` (root mean squared error), plus the scalar `s_ba2` that weights the chronological-age prior.

In [5]:
with open("../bioage_params/kdmage.json") as handle:
    params = json.load(handle)

params["features"]

['forced_expiratory_volume',
 'systolic_blood_pressure',
 'total_cholesterol',
 'hemoglobin_a1c',
 'albumin',
 'creatinine',
 'c_reactive_protein',
 'alkaline_phosphatase',
 'blood_urea_nitrogen',
 'age',
 'female']

## Load features

In [6]:
model.features = params["features"]
model.features

['forced_expiratory_volume',
 'systolic_blood_pressure',
 'total_cholesterol',
 'hemoglobin_a1c',
 'albumin',
 'creatinine',
 'c_reactive_protein',
 'alkaline_phosphatase',
 'blood_urea_nitrogen',
 'age',
 'female']

#### Normal feature ranges

Every feature above is registered in `pyaging`'s feature range registry, which is the single source of truth for units and plausible bounds; `model.feature_units` stays `None` so the registry is not shadowed by a stale copy. `pya.utils.get_feature_ranges("kdmage")` reports them for a saved clock.

Two of these deserve a note:

- `forced_expiratory_volume` is in **litres** (BioAge's `fev` in mL was rescaled during extraction), and the remaining biomarkers were fit on BioAge's SI-unit variants, so no post-hoc unit conversion is applied anywhere in this clock.
- `c_reactive_protein` is supplied **raw, in mg/dL**. BioAge's `lncrp` biomarker is `log1p(CRP in mg/dL)` — not the natural log that `phenoage` uses — so `KDMAge.preprocess` applies `log1p` internally and `q`/`k`/`s` for that biomarker remain on the `log1p` scale. A raw CRP is clamped to 0.01 mg/dL first, which is the registered lower bound, so a below-detection reading coded as `0` or an absent column cannot distort the estimate.

In [7]:
pd.DataFrame.from_records(
    pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
)

,feature,unit,low,high
0,forced_expiratory_volume,L,0.10,8.0
1,systolic_blood_pressure,mmHg,50.00,260.0
2,total_cholesterol,mmol/L,1.00,30.0
3,hemoglobin_a1c,%,2.00,20.0
4,albumin,g/L,10.00,70.0
5,creatinine,umol/L,10.00,3000.0
6,c_reactive_protein,mg/dL,0.01,50.0
7,alkaline_phosphatase,U/L,5.00,5000.0
8,blood_urea_nitrogen,mmol/L,0.50,60.0
9,age,years,0.00,122.5


## Load weights into base model

There is nothing to learn here: the Klemera-Doubal estimate is a closed form over the extracted parameters, so the base model is the identity and all of the arithmetic lives in `KDMAge.postprocess`. The parameters are stored as buffers, one set per sex.

`params[sex]["biomarkers"]` need not be in `model.features` order, so each vector is reindexed onto the feature order before it is stored.

In [8]:
model.base_model = torch.nn.Identity()

for sex in ["male", "female"]:
    fit = params[sex]
    order = [fit["biomarkers"].index(name) for name in model.features[:-2]]
    for key in ["q", "k", "s"]:
        setattr(model, f"{key}_{sex}", torch.tensor([fit[key][index] for index in order], dtype=torch.float64))
    setattr(model, f"s_ba2_{sex}", torch.tensor(fit["s_ba2"], dtype=torch.float64))

model.q_female

tensor([ 3.9277, 85.5114,  3.7846,  4.4498, 41.5707, 51.6685,  0.3033, 54.9584,
         2.2111], dtype=torch.float64)

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = "log1p_crp"
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = "klemera_doubal"
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Kwon, Dayoon, and Daniel W. Belsky. "A toolkit for '
             'quantification of biological age from blood chemistry and organ '
             'function test data: BioAge." GeroScience 43.6 (2021): 2795-2808.',
 'clock_name': 'kdmage',
 'data_type': 'clinical biomarkers',
 'doi': 'https://doi.org/10.1007/s11357-021-00480-5',
 'notes': 'Klemera-Doubal method trained sex-specifically on NHANES III (ages '
          '30-75, non-pregnant) using the BioAge package defaults. Biomarker '
          'parameters were fit on SI-unit variants so they are natively in '
          "pyaging's unit convention.",
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2020}
reference_values: None
preprocess_name: 'log1p_crp'
preprocess_dependencies: None
postprocess_name: 'klemera_doubal'
postprocess_dep

## Basic test

The smoke test feeds the midpoint of each feature's registered range, so the inputs are physiologically plausible rather than random.

In [13]:
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = torch.tensor(
    [[(record["low"] + record["high"]) / 2 for record in records]], dtype=torch.float64
)
model.eval()
model.to(torch.float64)
pred = model(midpoints)
pred

tensor([[419.3437]], dtype=torch.float64)

#### Parity with BioAge

The acceptance gate is `tests/predict/test_bioage_clocks.py`, which reproduces BioAge's own output for 20 NHANES IV subjects. Reproduced inline here as well, since a notebook that builds parameters should show that they land where the source package lands.

In [14]:
with open("../bioage_params/reference_predictions.json") as handle:
    reference = json.load(handle)

matrix = torch.tensor(
    [[row[name] for name in model.features] for row in reference["rows"]], dtype=torch.float64
)
with torch.inference_mode():
    predicted = model(matrix).squeeze(-1)

expected = torch.tensor(reference["expected"]["kdmage"], dtype=torch.float64)
print("max absolute difference:", (predicted - expected).abs().max().item())

max absolute difference: 6.863842827442568e-12


## Save torch model

In [15]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [16]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)